
# Data Analysis with SQL and Python

## Project: Delivery Time Deviation Prediction in Logistics

This notebook performs SQL and Python-based data analysis for the logistics dataset.

The original analysis requirements are:

1. Calculate average delivery time by distance, hour, and vehicle.
2. Analyze late delivery rate by zone.
3. Query correlation between distance and duration.
4. Aggregate by popular routes.

Because the selected dataset does not contain direct columns such as `distance`, `vehicle_type`, `delivery_zone`, `route_id`, or actual `delivery_duration`, this notebook creates meaningful proxy variables from the available dataset.

## Proxy Variable Mapping

| Original Requirement | Direct Column Available? | Proxy Used |
|---|---:|---|
| Distance | No | `distance_proxy` from `eta_variation_hours` |
| Vehicle | No | `vehicle_proxy` from `driver_behavior_score` and `fuel_consumption_rate` |
| Zone | No | `gps_zone` from GPS latitude and longitude bins |
| Route | No | `route_proxy` from `gps_zone` and `route_risk_group` |
| Delivery duration | No | `final_delivery_time_hours` from lead time, loading/unloading time, customs clearance time, and ETA variation |
| Late delivery | No direct flag | `is_late` from `delivery_time_deviation > 0` |

This makes the analysis compatible with the available dataset while still following the original analytical goals.



## 1. Setup and Load Dataset


In [ ]:

import pandas as pd
import numpy as np
import sqlite3
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

pd.set_option("display.max_columns", None)

# Load cleaned dataset if available, otherwise load original dataset
cleaned_file = Path("cleaned_dynamic_supply_chain_logistics_dataset.csv")
original_file = Path("dynamic_supply_chain_logistics_dataset.csv")

if cleaned_file.exists():
    df = pd.read_csv(cleaned_file)
    print("Loaded cleaned dataset.")
elif original_file.exists():
    df = pd.read_csv(original_file)
    print("Loaded original dataset.")
else:
    raise FileNotFoundError("Dataset file not found. Please place the CSV file in the same folder as this notebook.")

print("Dataset shape:", df.shape)
display(df.head())



## 2. Prepare Data and Create Proxy Variables

This section creates variables needed to match the SQL analysis requirements.

Important note:
- The dataset does not contain direct `distance`, `vehicle_type`, `delivery_zone`, or `route_id`.
- Therefore, proxy variables are created and clearly documented.


In [ ]:

# Convert timestamp to datetime
df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")

# Create hour feature
df["hour_of_day"] = df["timestamp"].dt.hour

# Create late delivery flag
# If delivery_time_deviation > 0, the delivery is later than expected
df["is_late"] = (df["delivery_time_deviation"] > 0).astype(int)

# Distance proxy
# Since actual distance is not available, eta_variation_hours is used as a proxy for delivery distance/time variation
df["distance_proxy"] = df["eta_variation_hours"]

# Distance groups for SQL aggregation
df["distance_group"] = pd.qcut(
    df["distance_proxy"],
    q=5,
    labels=["Very Short", "Short", "Medium", "Long", "Very Long"],
    duplicates="drop"
)

# Vehicle proxy
# Since vehicle type is not available, driver behavior and fuel consumption are used to create a vehicle/driver performance proxy
df["vehicle_score_proxy"] = (
    df["driver_behavior_score"] * 0.6 +
    (100 - df["fuel_consumption_rate"]) * 0.4
)

df["vehicle_proxy"] = pd.qcut(
    df["vehicle_score_proxy"],
    q=4,
    labels=["Low Performance", "Medium Performance", "Good Performance", "High Performance"],
    duplicates="drop"
)

# GPS-based zone
# Since delivery_zone is not available, create zone from latitude and longitude bins
df["lat_bin"] = pd.cut(df["vehicle_gps_latitude"], bins=5, labels=False)
df["long_bin"] = pd.cut(df["vehicle_gps_longitude"], bins=5, labels=False)
df["gps_zone"] = df["lat_bin"].astype(str) + "_" + df["long_bin"].astype(str)

# Route risk group
df["route_risk_group"] = pd.qcut(
    df["route_risk_level"],
    q=5,
    labels=["Very Low Risk", "Low Risk", "Medium Risk", "High Risk", "Very High Risk"],
    duplicates="drop"
)

# Route proxy
# Since route_id is not available, combine GPS zone and route risk group
df["route_proxy"] = df["gps_zone"].astype(str) + "_" + df["route_risk_group"].astype(str)

# Delivery duration proxy
# Since actual delivery_duration is not available, create a proxy using available time-related logistics variables
df["final_delivery_time_hours"] = (
    df["lead_time_days"] * 24 +
    df["loading_unloading_time"] +
    df["customs_clearance_time"] +
    df["eta_variation_hours"]
)

# Remove invalid duration proxy if any
df = df[df["final_delivery_time_hours"] > 0].reset_index(drop=True)

display(df[[
    "timestamp",
    "hour_of_day",
    "distance_proxy",
    "distance_group",
    "vehicle_proxy",
    "gps_zone",
    "route_proxy",
    "final_delivery_time_hours",
    "delivery_time_deviation",
    "is_late"
]].head())



## 3. Create SQLite Database

The dataframe is saved into a SQLite database table named `logistics`.


In [ ]:

conn = sqlite3.connect("logistics_analysis.db")

df.to_sql("logistics", conn, if_exists="replace", index=False)

def run_query(query):
    return pd.read_sql_query(query, conn)

print("SQLite database created successfully.")
print("Table name: logistics")



## 4. SQL Query 1: Average Delivery Time by Distance, Hour, and Vehicle

Original requirement:

> Calculate average delivery time by distance, hour, and vehicle.

Adapted implementation:

- `distance_group` is used as distance proxy.
- `hour_of_day` is extracted from timestamp.
- `vehicle_proxy` is created from driver behavior and fuel consumption.
- `final_delivery_time_hours` is used as delivery duration proxy.


In [ ]:

query_1 = '''
SELECT
    distance_group,
    hour_of_day,
    vehicle_proxy,
    COUNT(*) AS total_records,
    ROUND(AVG(final_delivery_time_hours), 2) AS avg_delivery_time_hours,
    ROUND(AVG(delivery_time_deviation), 2) AS avg_delivery_time_deviation
FROM logistics
GROUP BY distance_group, hour_of_day, vehicle_proxy
ORDER BY avg_delivery_time_hours DESC
LIMIT 20;
'''

avg_delivery_by_distance_hour_vehicle = run_query(query_1)
display(avg_delivery_by_distance_hour_vehicle)



### Interpretation

This query shows how average delivery time changes across distance groups, hours of the day, and vehicle/driver performance groups. It helps identify operational conditions that are associated with longer delivery time.



## 5. SQL Query 2: Late Delivery Rate by Zone and Hour

Original requirement:

> Analyze late delivery rate by zone.

Adapted implementation:

- `gps_zone` is created from GPS latitude and longitude.
- `is_late` is created from `delivery_time_deviation > 0`.
- The query calculates late delivery rate by zone and hour.


In [ ]:

query_2 = '''
SELECT
    gps_zone,
    hour_of_day,
    COUNT(*) AS total_records,
    SUM(is_late) AS late_records,
    ROUND(AVG(is_late) * 100, 2) AS late_delivery_rate_percent,
    ROUND(AVG(delivery_time_deviation), 2) AS avg_delivery_time_deviation
FROM logistics
GROUP BY gps_zone, hour_of_day
ORDER BY late_delivery_rate_percent DESC
LIMIT 30;
'''

late_rate_by_zone_hour = run_query(query_2)
display(late_rate_by_zone_hour)



### Interpretation

This query identifies GPS-based zones and hours with high late delivery rates. It is useful for detecting time-location combinations where ETA errors are more likely to occur.



## 6. SQL Query 3: Correlation Between Distance and Duration

Original requirement:

> Query correlation between distance and duration.

Adapted implementation:

- `distance_proxy` is used instead of actual distance.
- `final_delivery_time_hours` is used instead of actual delivery duration.
- SQLite does not always support a built-in `CORR()` function, so SQL is used to extract the required columns and Pandas is used to calculate Pearson correlation.


In [ ]:

query_3 = '''
SELECT
    distance_proxy,
    final_delivery_time_hours,
    delivery_time_deviation
FROM logistics;
'''

corr_data = run_query(query_3)

corr_distance_duration = corr_data["distance_proxy"].corr(corr_data["final_delivery_time_hours"])
corr_distance_deviation = corr_data["distance_proxy"].corr(corr_data["delivery_time_deviation"])

print("Correlation between distance_proxy and final_delivery_time_hours:", corr_distance_duration)
print("Correlation between distance_proxy and delivery_time_deviation:", corr_distance_deviation)



### Interpretation

The correlation value helps measure the strength and direction of the relationship between estimated distance/time variation and delivery duration. A positive value indicates that higher distance proxy values are associated with longer delivery time or larger ETA deviation.



## 7. SQL Query 4: Aggregate by Popular Routes

Original requirement:

> Aggregate by popular routes.

Adapted implementation:

- `route_proxy` is created by combining GPS zone and route risk group.
- Popular routes are defined as route proxies with the highest number of records.


In [ ]:

query_4 = '''
SELECT
    route_proxy,
    COUNT(*) AS total_records,
    ROUND(AVG(final_delivery_time_hours), 2) AS avg_delivery_time_hours,
    ROUND(AVG(delivery_time_deviation), 2) AS avg_delivery_time_deviation,
    ROUND(AVG(delay_probability), 4) AS avg_delay_probability,
    ROUND(AVG(traffic_congestion_level), 2) AS avg_traffic_congestion_level
FROM logistics
GROUP BY route_proxy
ORDER BY total_records DESC
LIMIT 10;
'''

popular_routes = run_query(query_4)
display(popular_routes)



### Interpretation

This query shows the most common route proxies and summarizes their average delivery time, ETA deviation, delay probability, and traffic congestion level. It supports route-level logistics analysis even when direct route IDs are not available.



## 8. Additional SQL Analysis: High-Risk Delivery Records

This additional query helps identify records with high delay probability and high delivery deviation.


In [ ]:

query_5 = '''
SELECT
    timestamp,
    gps_zone,
    route_proxy,
    traffic_congestion_level,
    route_risk_level,
    delay_probability,
    risk_classification,
    final_delivery_time_hours,
    delivery_time_deviation
FROM logistics
WHERE delay_probability >= 0.8
ORDER BY delivery_time_deviation DESC
LIMIT 10;
'''

high_risk_records = run_query(query_5)
display(high_risk_records)



## 9. Python Visualization for SQL Results


In [ ]:

# Bar chart: average delivery time by distance group
distance_summary = df.groupby("distance_group")["final_delivery_time_hours"].mean().reset_index()

plt.figure(figsize=(8, 5))
sns.barplot(data=distance_summary, x="distance_group", y="final_delivery_time_hours")
plt.title("Average Delivery Time by Distance Proxy Group")
plt.xlabel("Distance Proxy Group")
plt.ylabel("Average Delivery Time (Hours)")
plt.xticks(rotation=45)
plt.show()


In [ ]:

# Heatmap: late delivery rate by GPS zone and hour
zone_hour = df.groupby(["gps_zone", "hour_of_day"])["is_late"].mean().reset_index()
zone_hour["late_rate_percent"] = zone_hour["is_late"] * 100

heatmap_data = zone_hour.pivot(index="gps_zone", columns="hour_of_day", values="late_rate_percent")

plt.figure(figsize=(14, 8))
sns.heatmap(heatmap_data, cmap="Reds")
plt.title("Late Delivery Rate by GPS Zone and Hour")
plt.xlabel("Hour of Day")
plt.ylabel("GPS-Based Zone")
plt.show()


In [ ]:

# Scatter plot: distance proxy vs delivery duration proxy
plt.figure(figsize=(8, 5))
sns.scatterplot(
    data=df,
    x="distance_proxy",
    y="final_delivery_time_hours",
    alpha=0.4
)

plt.title("Distance Proxy vs Delivery Time")
plt.xlabel("Distance Proxy")
plt.ylabel("Final Delivery Time (Hours)")
plt.show()



## 10. Summary of SQL and Python Analysis

The analysis was adapted to the available logistics dataset. Since the dataset does not contain direct distance, vehicle type, delivery zone, route ID, or delivery duration columns, proxy variables were created.

The SQL analysis successfully addresses the original requirements by:

1. Calculating average delivery time by distance proxy, hour, and vehicle proxy.
2. Analyzing late delivery rate by GPS-based zone and hour.
3. Calculating correlation between distance proxy and delivery duration proxy.
4. Aggregating logistics records by popular route proxies.

These results provide useful insights into delivery performance and prepare the dataset for further visualization and machine learning modeling.
